[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C26_Frontier_Agents_Course/04_agentic_rl/04_agentic_rl.ipynb)

# 04 · Agentic RL 与编码 agent

目标：用**纯 numpy** 从零写一个**玩具 agentic RL**——可验证奖励 → 轨迹采样 → 折扣回报 → **REINFORCE** 策略梯度更新 → 优势基线降方差 → 拒绝采样与 pass@k，全程 `assert` 验证、**无需 API key**。

路线：可验证奖励环境 → softmax 策略采样 → 折扣回报 → REINFORCE 更新(看奖励上升) → 基线降方差 → 拒绝采样/pass@k → ✏️ 练习 → 📖 答案 → 🧪 SWE-bench 式 pass@k 胶囊。

> 心智模型：**对成功的整条轨迹「点赞」，把赞顺着轨迹分给各步动作，让以后更可能重走成功的路**。难点是末尾一个赞如何公平分给中间几十步——信用分配。

## 1 · 带可验证奖励的玩具「编码」任务

RL 的燃料是**可验证奖励**：跑测试，过了给 1、否则 0。我们造一个极小的「选对修改」任务：
每个状态下有几个候选改动（动作），其中恰有一个能让「测试」通过。`run_tests` 就是我们的客观裁判。

这是一个单步 MDP（contextual bandit）——把机制看清；真实里把它换成几十步、几十亿参数即可。

In [ ]:
import numpy as np

# 一个玩具「代码库」：每个任务=一个状态 s，动作=选第几个候选补丁；
# 只有 correct_patch[s] 能让单元测试通过。run_tests 是可验证奖励。
N_STATES = 3            # 3 个 issue
N_ACTIONS = 4           # 每个 issue 有 4 个候选补丁
CORRECT_PATCH = np.array([2, 0, 3])     # 每个 issue 的正确补丁下标(我们设定的真相)

def run_tests(state, action):
    '''可验证奖励：选对补丁则测试通过(1)，否则(0)。客观、可程序化、难作弊。'''
    return 1 if action == CORRECT_PATCH[state] else 0

# 验证裁判正确
assert run_tests(0, 2) == 1 and run_tests(0, 1) == 0
assert run_tests(2, 3) == 1 and run_tests(2, 0) == 0
# 一个乱选的 agent 的期望成功率 = 1/N_ACTIONS
rng = np.random.default_rng(0)
rand_acts = rng.integers(0, N_ACTIONS, size=3000)
rand_states = rng.integers(0, N_STATES, size=3000)
rand_success = np.mean([run_tests(s, a) for s, a in zip(rand_states, rand_acts)])
print(f'随机策略成功率 ≈ {rand_success:.3f} (理论 {1/N_ACTIONS:.3f})')
assert abs(rand_success - 1/N_ACTIONS) < 0.05
print('✅ 可验证奖励就位：测试通过=1，乱选基线≈1/4。RL 要把它从 0.25 推高')

## 2 · softmax 策略 + 轨迹采样

策略 π(a|s) = softmax(theta[s])，theta 是一张 `[N_STATES, N_ACTIONS]` 的 logits 表（代替神经网络）。
**采样一条轨迹**：给定状态，按 π 抽一个动作，跑测试拿奖励。返回 (state, action, reward)。

In [ ]:
def softmax(x):
    x = x - x.max()                  # 数值稳定
    e = np.exp(x)
    return e / e.sum()

def policy_probs(theta, s):
    return softmax(theta[s])

def sample_action(theta, s, rng):
    p = policy_probs(theta, s)
    return int(rng.choice(len(p), p=p))

def rollout(theta, s, rng):
    '''单步轨迹：在状态 s 采一个动作，跑测试拿奖励。返回 (s, a, r)。'''
    a = sample_action(theta, s, rng)
    r = run_tests(s, a)
    return s, a, r

theta0 = np.zeros((N_STATES, N_ACTIONS))    # 初始：每个动作等概率
p0 = policy_probs(theta0, 0)
print('初始策略 π(·|s=0):', np.round(p0, 3))
assert np.allclose(p0, 0.25), '全 0 logits 应给均匀分布'
# 初始平均奖励 ≈ 0.25
rng = np.random.default_rng(1)
init_rewards = [rollout(theta0, int(rng.integers(0, N_STATES)), rng)[2] for _ in range(3000)]
print(f'初始策略平均奖励 ≈ {np.mean(init_rewards):.3f}')
assert abs(np.mean(init_rewards) - 0.25) < 0.05
print('✅ softmax 策略 + 采样就位；未训练时与随机相当(0.25)')

## 3 · 折扣回报 $G_t=\sum_k \gamma^k r_{t+k}$

一个动作的好坏看它**之后**引出的全部奖励。折扣因子 γ∈(0,1] 让近期奖励权重更高。
用从后往前递推 `G_t = r_t + γ·G_{t+1}` 计算，并与手算对照。

In [ ]:
def discounted_returns(rewards, gamma=0.9):
    '''给一条轨迹的奖励序列，返回每一步的折扣回报 G_t。从后往前 O(T) 递推。'''
    G = np.zeros(len(rewards), dtype=float)
    running = 0.0
    for t in reversed(range(len(rewards))):
        running = rewards[t] + gamma * running
        G[t] = running
    return G

# 手算对照：rewards=[0,0,1], gamma=0.9
# G2 = 1; G1 = 0 + 0.9*1 = 0.9; G0 = 0 + 0.9*0.9 = 0.81
G = discounted_returns([0, 0, 1], gamma=0.9)
print('G =', np.round(G, 4))
assert np.allclose(G, [0.81, 0.9, 1.0]), '与手算一致'
# gamma=1 时退化成 reward-to-go(从 t 起的奖励和)
G1 = discounted_returns([1, 1, 1], gamma=1.0)
assert np.allclose(G1, [3, 2, 1])
print('gamma=1 (reward-to-go):', G1)
print('✅ 折扣回报正确：越靠近末尾奖励，分到的回报越大 —— 朴素信用分配')

## 4 · REINFORCE：让高回报动作更可能（看奖励上升）

策略梯度：`∇logπ(a|s) = onehot(a) - softmax(theta[s])`，更新 `theta[s] += lr · G · ∇logπ`。
直觉：把**实际选到、且回报高**的动作往上推。训练若干轮，应看到**平均奖励从 0.25 升到接近 1**、最优动作概率上升。

In [ ]:
def grad_logp(theta, s, a):
    '''softmax 策略对 logits 的对数似然梯度: onehot(a) - π(·|s)。'''
    p = policy_probs(theta, s)
    g = -p.copy()
    g[a] += 1.0
    return g

def train_reinforce(theta, rng, n_iters=2000, batch=16, lr=0.5, gamma=1.0):
    theta = theta.copy()
    history = []
    for it in range(n_iters):
        grad = np.zeros_like(theta)
        batch_reward = 0.0
        for _ in range(batch):
            s = int(rng.integers(0, N_STATES))
            s, a, r = rollout(theta, s, rng)
            G = r                       # 单步任务: 回报=奖励
            grad[s] += G * grad_logp(theta, s, a)   # 累积策略梯度
            batch_reward += r
        theta += lr * grad / batch      # 上升一步(最大化期望回报)
        history.append(batch_reward / batch)
    return theta, history

rng = np.random.default_rng(2)
theta_trained, hist = train_reinforce(theta0, rng)
early = np.mean(hist[:50]); late = np.mean(hist[-200:])
print(f'训练初期平均奖励 ≈ {early:.3f}  ->  训练后期 ≈ {late:.3f}')
# 训练后，每个状态最高概率的动作应当就是正确补丁
learned = np.array([policy_probs(theta_trained, s).argmax() for s in range(N_STATES)])
print('学到的最优动作:', learned, ' 真相:', CORRECT_PATCH)
assert late > 0.85, 'REINFORCE 后平均奖励应显著上升(>0.85)'
assert np.array_equal(learned, CORRECT_PATCH), '策略应收敛到正确补丁'
print('✅ REINFORCE 把成功率从 0.25 推到 >0.85，策略收敛到正确补丁')

## 5 · 基线 / 优势：降低梯度方差

REINFORCE 方差大、收敛慢。减去一个**与动作无关**的基线 b（如批次平均回报）得到优势 A=G−b：
**期望不变（无偏）、方差变小**。我们直接验证这个经典结论。

In [ ]:
def grad_samples(theta, s, rng, n=6000, baseline=0.0):
    '''在状态 s 采 n 条单步轨迹，返回每条的「整条策略梯度向量」 (r-baseline)*∇logπ(a|s)。
       形状 [n, N_ACTIONS]。我们量它的总方差(各坐标方差之和)，这正是基线定理约束的量。'''
    gs = np.zeros((n, N_ACTIONS))
    for i in range(n):
        _, a, r = rollout(theta, s, rng)
        gs[i] = (r - baseline) * grad_logp(theta, s, a)
    return gs

def total_variance(gs):
    '''总方差 = 各坐标方差之和 = E||g - E g||^2，基线定理保证它被无偏地降低。'''
    return float(gs.var(axis=0).sum())

# 用一个非平凡(已偏向正确)的策略来比方差；在 state 0 上比较
theta_mid = np.zeros((N_STATES, N_ACTIONS))
for s in range(N_STATES):
    theta_mid[s, CORRECT_PATCH[s]] = 0.8        # 略偏向正确补丁，π(correct)≈0.4
s0 = 0
p_correct = policy_probs(theta_mid, s0)[CORRECT_PATCH[s0]]

# 无基线
rng = np.random.default_rng(3)
g_no = grad_samples(theta_mid, s0, rng, baseline=0.0)
# 有基线：b = 该状态的平均回报(=单次成功率)，是接近最优的常数基线
rng = np.random.default_rng(3)                  # 同种子，公平对比
b = p_correct                                  # 0/1 奖励下平均回报就是成功率
g_with = grad_samples(theta_mid, s0, rng, baseline=b)

var_no, var_with = total_variance(g_no), total_variance(g_with)
mean_no, mean_with = g_no.mean(axis=0), g_with.mean(axis=0)
print(f'基线值 b ≈ {b:.3f}')
print(f'无基线 总方差 = {var_no:.5f}')
print(f'有基线 总方差 = {var_with:.5f}  (降低 {100*(1-var_with/var_no):.0f}%)')
print(f'梯度均值(方向) 无基线={np.round(mean_no,3)}  有基线={np.round(mean_with,3)}')
# 方差严格下降
assert var_with < var_no, '加基线应降低梯度总方差'
# 期望(方向)几乎不变：减去常数基线是无偏的(E[b·∇logπ]=0)
assert np.allclose(mean_no, mean_with, atol=0.03), '减常数基线不应改变梯度期望(无偏)'
print('✅ 优势基线：总方差显著下降，而梯度方向(期望)几乎不变 —— 现代 RL(PPO/GRPO)的核心组件')

## 6 · 拒绝采样与 pass@k

RL 之外的简单路线：对一个任务采 k 次、用测试**只留成功的**、拿去自举。
其数学基础是 **pass@k = 1−(1−p)^k**（单次成功率 p）。我们验证解析式 = 经验估计，并做一次拒绝采样自举。

In [ ]:
def pass_at_k_analytic(p, k):
    return 1.0 - (1.0 - p) ** k

def pass_at_k_empirical(theta, s, k, rng, n_tasks=4000):
    '''经验 pass@k：重复 n_tasks 次「采 k 条、有一条成功就算过」，取比例。'''
    succ = 0
    for _ in range(n_tasks):
        ok = any(rollout(theta, s, rng)[2] == 1 for _ in range(k))
        succ += int(ok)
    return succ / n_tasks

# 用一个单次成功率 p≈0.4 的策略(在 state 0 上)
theta_p = theta0.copy()
# 令 π(correct|0) ≈ 0.4：correct logit 比其它高一些
from math import log
theta_p[0, :] = 0.0
theta_p[0, CORRECT_PATCH[0]] = log(0.4 / 0.2)   # 解析设定，使 correct 概率≈0.4
p_single = policy_probs(theta_p, 0)[CORRECT_PATCH[0]]
print(f'单次成功率 p ≈ {p_single:.3f}')

rng = np.random.default_rng(4)
for k in [1, 3, 5]:
    ana = pass_at_k_analytic(p_single, k)
    emp = pass_at_k_empirical(theta_p, 0, k, rng)
    print(f'pass@{k}: 解析={ana:.3f}  经验={emp:.3f}')
    assert abs(ana - emp) < 0.04, f'pass@{k} 解析与经验应接近'
# 拒绝采样自举：采 k 条、只用成功的动作做一次「加强」
print('\n--- 拒绝采样自举(只用成功轨迹加强) ---')
rng = np.random.default_rng(5)
theta_rs = theta_p.copy()
before = policy_probs(theta_rs, 0)[CORRECT_PATCH[0]]
for _ in range(300):
    succ_actions = [a for (_, a, r) in (rollout(theta_rs, 0, rng) for _ in range(8)) if r == 1]
    for a in succ_actions:                       # 加强成功动作(SFT 式)
        theta_rs[0] += 0.3 * grad_logp(theta_rs, 0, a)
after = policy_probs(theta_rs, 0)[CORRECT_PATCH[0]]
print(f'correct 动作概率: {before:.3f} -> {after:.3f}')
assert after > before + 0.2, '拒绝采样自举应提升成功率'
print('✅ pass@k=1-(1-p)^k 验证通过；拒绝采样把单次成功率显著抬高')

---
## ✏️ 练习 1：reward-to-go vs 整条回报赋给每步

信用分配的两种朴素做法：(a) **reward-to-go**——每步只拿它**之后**的奖励和 `G_t=Σ_{k≥t} r_k`；(b) **整条回报**——把整条轨迹总奖励赋给**每一步**。

实现 `reward_to_go(rewards)`（gamma=1 的折扣回报特例）。它比 (b) 更合理：早期动作不该为它之前就拿到的奖励负责。

In [ ]:
def reward_to_go(rewards):
    # TODO: 返回每步的「之后奖励和」 G_t = sum(rewards[t:])
    #       提示：就是 gamma=1 的折扣回报；可从后往前累加
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
assert list(reward_to_go([1, 0, 2, 0, 3])) == [6, 5, 5, 3, 3]
assert list(reward_to_go([0, 0, 1])) == [1, 1, 1]
# 与「整条回报赋给每步」对比：后者会给每步都赋 total，对早期步不公平
total = sum([1, 0, 2, 0, 3])
naive = [total] * 5
assert reward_to_go([1, 0, 2, 0, 3])[0] == total      # 第0步两者相同
assert reward_to_go([1, 0, 2, 0, 3])[-1] < naive[-1]  # 但末步 reward-to-go 更小(更合理)
print('reward-to-go:', list(reward_to_go([1, 0, 2, 0, 3])))
print('✅ 练习 1 通过：reward-to-go 让每步只为它之后的奖励负责')

## ✏️ 练习 2：一步 REINFORCE 梯度指向被奖励的动作

验证策略梯度的方向：对一个均匀策略，若动作 a 得了正回报 G>0，则一步更新后 **π(a|s) 应当上升**。

实现 `one_reinforce_step(theta, s, a, G, lr)`：返回更新后的 theta（`theta[s] += lr·G·∇logπ(a|s)`，∇logπ=onehot(a)−π）。

In [ ]:
def one_reinforce_step(theta, s, a, G, lr=0.5):
    # TODO: 复制 theta；对 theta[s] 加上 lr*G*(onehot(a)-softmax(theta[s]))；返回新 theta
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
th = np.zeros((N_STATES, N_ACTIONS))
p_before = policy_probs(th, 0)[1]
th2 = one_reinforce_step(th, s=0, a=1, G=1.0, lr=0.5)
p_after = policy_probs(th2, 0)[1]
assert p_after > p_before, '正回报应抬高该动作概率'
# 负回报应压低
th3 = one_reinforce_step(th, s=0, a=1, G=-1.0, lr=0.5)
assert policy_probs(th3, 0)[1] < p_before, '负回报应压低该动作概率'
# 不改变其它状态
assert np.allclose(th2[1], th[1]) and np.allclose(th2[2], th[2])
print(f'π(a=1|s=0): {p_before:.3f} --(G=+1)--> {p_after:.3f}')
print('✅ 练习 2 通过：梯度把被正奖励的动作往上推、被负奖励的往下压')

## ✏️ 练习 3：pass@k 解析式 + 经验估计

实现两个版本并验证一致：
(a) `pak_analytic(p, k)` = 1−(1−p)^k；
(b) `pak_empirical(successes_matrix)`：给一个 `[n_tasks, k]` 的 0/1 矩阵（每个任务 k 次尝试的成败），返回经验 pass@k（每行只要有一个 1 就算该任务通过，取行通过比例）。

In [ ]:
def pak_analytic(p, k):
    # TODO: 1 - (1-p)**k
    raise NotImplementedError

def pak_empirical(successes_matrix):
    # TODO: successes_matrix 形状 [n_tasks, k]，每行 any()>0 算通过；返回通过比例(float)
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
assert abs(pak_analytic(0.5, 1) - 0.5) < 1e-9
assert abs(pak_analytic(0.5, 2) - 0.75) < 1e-9      # 1-(0.5)^2
assert abs(pak_analytic(0.2, 5) - (1 - 0.8**5)) < 1e-9
# 经验：构造一个 p≈0.3 的成败矩阵，k=4，经验 pass@4 应≈解析
rng = np.random.default_rng(7)
p_true, k = 0.3, 4
mat = (rng.random((6000, k)) < p_true).astype(int)
emp = pak_empirical(mat)
ana = pak_analytic(p_true, k)
print(f'pass@{k}: 解析={ana:.3f}  经验={emp:.3f}')
assert abs(emp - ana) < 0.03
print('✅ 练习 3 通过：pass@k 解析式与经验估计一致')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def reward_to_go(rewards):
    out = np.zeros(len(rewards), dtype=float)
    running = 0.0
    for t in reversed(range(len(rewards))):
        running = rewards[t] + running       # gamma=1
        out[t] = running
    return out

In [ ]:
# 练习 2 参考答案
def one_reinforce_step(theta, s, a, G, lr=0.5):
    theta = theta.copy()
    theta[s] += lr * G * grad_logp(theta, s, a)
    return theta

In [ ]:
# 练习 3 参考答案
def pak_analytic(p, k):
    return 1.0 - (1.0 - p) ** k

def pak_empirical(successes_matrix):
    m = np.asarray(successes_matrix)
    return float((m.sum(axis=1) > 0).mean())

---
## 🧪 真实数据胶囊：SWE-bench 式 pass@k

下面用**贴近真实**的「每个任务的单次成功率 p」（量级参考真实编码 agent 在 SWE-bench 上的报告：单题成功率从难到易差异很大），计算 pass@1 vs pass@k，体会**「有验证器(测试)就能用 best-of-n 把成功率抬上去」**——这正是把 pass@k 变成可交付能力的关键。

> 真实里：跑 k 次、用单元测试挑出通过的那次提交，等效成功率就是 pass@k。

In [ ]:
# 几个任务的单次成功率 p（量级贴近真实编码 agent 的逐题难度分布）
TASK_P = {
    'easy-typo-fix':      0.70,
    'medium-logic-bug':   0.35,
    'hard-multifile':     0.12,
    'very-hard-refactor': 0.04,
}

def pak(p, k):
    return 1.0 - (1.0 - p) ** k

print(f"{'task':22s}{'p(=pass@1)':>12s}{'pass@5':>9s}{'pass@10':>9s}")
for name, p in TASK_P.items():
    print(f'{name:22s}{p:>12.2f}{pak(p,5):>9.2f}{pak(p,10):>9.2f}')
# 即便很难的任务(p=0.04)，pass@10 也比 pass@1 高一截 —— 验证器让多采样变现
assert pak(0.04, 10) > 0.04 * 3, 'best-of-10 应显著高于单次'
assert pak(0.35, 5) > 0.85, '中等任务 pass@5 应很高'
print('\n观察：有了测试当验证器，best-of-n 把单次成功率放大成可交付的 pass@k')

**🧪 胶囊练习**：实现 `pass_at_k_table(task_p, ks)`：给定 `{任务名: p}` 与一组 k 值，返回 `{任务名: {k: pass@k}}` 的嵌套字典（pass@k=1−(1−p)^k）。

In [ ]:
def pass_at_k_table(task_p, ks=(1, 3, 5, 10)):
    # TODO: 返回 {name: {k: 1-(1-p)**k for k in ks} for name, p in task_p.items()}
    raise NotImplementedError

In [ ]:
# 自测
tbl = pass_at_k_table(TASK_P, ks=(1, 3, 5, 10))
assert abs(tbl['easy-typo-fix'][1] - 0.70) < 1e-9        # pass@1 == p
assert abs(tbl['hard-multifile'][10] - (1 - 0.88**10)) < 1e-9
# pass@k 对 k 单调不减
for name in TASK_P:
    vals = [tbl[name][k] for k in (1, 3, 5, 10)]
    assert all(vals[i] <= vals[i+1] + 1e-12 for i in range(len(vals)-1))
print('hard-multifile 的 pass@k:', {k: round(tbl['hard-multifile'][k], 3) for k in (1,3,5,10)})
print('✅ 胶囊练习通过')

In [ ]:
# 📖 胶囊参考答案
def pass_at_k_table(task_p, ks=(1, 3, 5, 10)):
    return {name: {k: 1.0 - (1.0 - p) ** k for k in ks}
            for name, p in task_p.items()}

---
## 🔧 旁注：玩具 REINFORCE 如何对应真实 agentic RL

本课的玩具回路，放大到真实编码 agent 训练就是（概念伪代码，**本环境不跑，需大模型 + GPU**）：

```python
# 真实 agentic RL（如 SWE-bench 上训练编码 agent）的骨架
for step in range(N):
    batch = []
    for task in sample_tasks():                 # 真实 GitHub issue + 测试
        traj = run_agent(policy_llm, task)       # 跑完整 agent 轨迹(几十步工具调用)
        reward = run_unit_tests(traj.patch)      # 可验证奖励: 测试全过=1
        batch.append((traj, reward))
    # 用策略梯度(实践中是 PPO/GRPO，不是朴素 REINFORCE)更新 policy_llm 的权重
    advantages = compute_advantages(batch)      # G - baseline，本课讲的优势
    ppo_update(policy_llm, batch, advantages)    # 内核仍是 ∇logπ · A
```

对应关系：我们的 `theta` ↔ LLM 的几十亿参数、`run_tests` ↔ `run_unit_tests`(SWE-bench)、`rollout` ↔ 一条完整 agent 轨迹、REINFORCE+基线 ↔ PPO/GRPO。**内核数学一模一样**，只是规模与稳定化手段不同。

### 小结
- agentic RL = 在 agent 的多步轨迹上做 RL，让它**越用越强**(而非只靠提示)。
- **可验证奖励**(测试通过=1)是能 scale 的前提：客观、便宜、难作弊(SWE-bench)。
- **折扣回报** G_t=Σγ^k r：越靠近成功的动作分到的回报越大(朴素信用分配)。
- **信用分配**是 RL 难的根本：末尾一个奖励如何公平分给中间几十步。
- **REINFORCE**: theta += lr·G·(onehot(a)−π)；**优势基线** A=G−b 降方差、不改期望。
- **拒绝采样 / pass@k=1−(1−p)^k**：有验证器就能用 best-of-n 把成功率变现，再自举。

下一站：**模块 05 · Agent 评测与安全** —— 会做(pass@k) ≠ 稳定做对(pass^k)，以及被劫持的防线。